# trainer-class-skeleton — ex3: Trainer with best-val-loss checkpoint snapshot

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `trainer-class-skeleton`. Running the final beacon cell reports progress against the `Trainer: Trainer class skeleton` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: Trainer class skeleton` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-class-skeleton`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-class-skeleton"
DD_SUBTOPIC = "Trainer: Trainer class skeleton"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Best-val-loss checkpoint — track best, snapshot state_dict

Ex1 implemented `fit`+`validate`+`_step`. Ex2 added a callback list. The deepening move adds the SINGLE most-common Trainer extension: remember the best validation loss seen so far, and snapshot the model's `state_dict()` when it improves.

```python
self.best_val_loss = float('inf')
self.best_state    = None    # snapshot dict (or None if never improved)

def validate(self):
    # ... compute val_loss ...
    self.history['val_loss'].append(val_loss)
    if val_loss < self.best_val_loss:
        self.best_val_loss = val_loss
        # DEEP-COPY: state_dict() returns LIVE references; we need a snapshot.
        self.best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
```

**Why deep-copy.** `model.state_dict()` returns a dict of LIVE tensors — the same tensors that `optimizer.step()` is about to mutate. Saving the dict directly would mean `best_state` updates IN LOCKSTEP with the training, defeating the whole point. `.detach().clone()` per tensor produces an independent snapshot.

**`best_state = None` sentinel.** Before the first validate call, no snapshot has been taken. Returning `None` is cleaner than 'snapshot of the random init' — callers can check `if trainer.best_state is None: ...`.

**Strict `<`, not `<=`.** Use strict less-than for 'improved'. With `<=`, ties would overwrite the snapshot needlessly and you'd snapshot the LATER tied model rather than the EARLIER one (which is usually preferred — earlier means less overfit).

### Exercise 3 — Trainer with best-val-loss checkpoint snapshot

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply best-val-loss tracking to the Trainer skeleton: maintain `self.best_val_loss` (float, init `inf`) and `self.best_state` (deep-copied state_dict snapshot) updated each time validation loss strictly improves.
> Keywords: checkpoint, best-val-loss, state-dict, trainer
> ```

**KCs targeted:** `best-val-loss-monotone-improvement-check`, `state-dict-snapshot-via-detach-clone`

Implement `Ex3TrainerWithCheckpoint`. Same shape as the minimal Trainer, plus best-val-loss tracking and a snapshot of `model.state_dict()` whenever the val loss improves.

1. `__init__(self, model, optimizer, train_loader, val_loader, loss_fn)`:
   - Store the five args.
   - `self.step = 0`, `self.history = {'train_loss': [], 'val_loss': []}`.
   - `self.best_val_loss = float('inf')`.
   - `self.best_state = None` (will become a dict of cloned tensors once validate runs at least once).

2. `_step(self, x, y)`: forward + loss only.

3. `fit(self, n_epochs)`: standard train loop calling `_step`, `loss.backward()`, `optimizer.step()`, `optimizer.zero_grad()`, `self.step += 1`, `self.history['train_loss'].append(loss.item())`. After each epoch, call `self.validate()`.

4. `validate(self)`: compute weighted-average val loss under `t.inference_mode()`. Append to `self.history['val_loss']`. Then:
   - If `val_loss < self.best_val_loss` (STRICT inequality):
     - `self.best_val_loss = val_loss`
     - `self.best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}`
   - Otherwise: do not touch `best_state`.

Snapshot policy: use `.detach().clone()` per tensor so the snapshot is INDEPENDENT of subsequent training mutations.

In [ ]:
class Ex3TrainerWithCheckpoint:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}
        self.best_val_loss = float('inf')
        self.best_state = None

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _epoch in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total = 0.0
        count = 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        val_loss = total / count
        self.history['val_loss'].append(val_loss)
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_state = {
                k: v.detach().clone() for k, v in self.model.state_dict().items()
            }


<details><summary>Solution</summary>

```python
class Ex3TrainerWithCheckpoint:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}
        self.best_val_loss = float('inf')
        self.best_state = None

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _epoch in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total = 0.0
        count = 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        val_loss = total / count
        self.history['val_loss'].append(val_loss)
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_state = {
                k: v.detach().clone() for k, v in self.model.state_dict().items()
            }
```

**`.detach().clone()` is the canonical snapshot idiom.** `clone()` alone preserves the autograd graph link; `detach()` alone shares storage with the live model. The combo gives an independent tensor that is NOT mutated by subsequent `optimizer.step()` calls.

**Strict `<` not `<=`.** With `<=`, ties on val loss overwrite the snapshot. The convention is 'keep the EARLIER best' — earlier means less overfit. Strict less-than enforces that.

**Sentinel `None` is cleaner than 'init from random init'.** Before the first `validate()` call, no real evaluation has happened. Storing a random-init snapshot would lie about 'best'; `None` lets callers detect 'no validation has run yet'.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()